In [0]:
!pip install xgboost
dbutils.library.restartPython()

In [0]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import *
from pyspark.sql.types import  *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.session import SparkSession
import joblib
# import plotly.express as px
# import plotly.io as pio

In [0]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics, feature_selection
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
import xgboost as xgb

In [0]:
spark= SparkSession.builder.appName("Fraud").getOrCreate()

In [0]:
test_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTest.csv", header=True)
train_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTrain.csv", header=True)

In [0]:
cols_to_drop = [
    "_c0",
    "first",
    "last",
    "street",
    "trans_num",
    "unix_time"
]

train_df = train_df.drop(*cols_to_drop)
test_df = test_df.drop(*cols_to_drop)

In [0]:
max_date= train_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

train_df= train_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
train_df= train_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
train_df= train_df.drop('dob')

In [0]:
max_date= test_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

test_df= test_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
test_df= test_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
test_df= test_df.drop('dob')

In [0]:
fraud_trans= train_df.filter(F.col('is_fraud')==1)
non_fraud_trans= train_df.filter(F.col('is_fraud')==0)
fraud_count= fraud_trans.count()
non_fraud_count= non_fraud_trans.count()
fraction= np.divide(fraud_count, non_fraud_count)
sampled_non_fraud= non_fraud_trans.sample(withReplacement=False, fraction=(fraction*1.5), seed=42)
balanced_df= fraud_trans.union(sampled_non_fraud)
balanced_df.groupBy('is_fraud').count().display()

In [0]:
X= balanced_df.drop('is_fraud','trans_date_trans_time').toPandas()
y= balanced_df.toPandas()['is_fraud']

In [0]:
X.head()

In [0]:
cat_cols= X.select_dtypes(include='object').columns.tolist()
num_cols= X.select_dtypes(exclude='object').columns.tolist()

tree_cat_pipeline= Pipeline([
    ('Ordinal encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

linear_cat_pipeline= Pipeline([
    ('OHE', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

num_pipeline= Pipeline([
    ('scaler', RobustScaler())
])

tree_prep= ColumnTransformer(
    [
        ('tree_cat', tree_cat_pipeline, cat_cols),
        ('tree_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

linear_prep= ColumnTransformer(
    [
        ('linear_cat', linear_cat_pipeline, cat_cols),
        ('linear_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

In [0]:
X_train, X_test, y_train, y_test= train_test_split(X,y,train_size=0.8,random_state=42,stratify=y)
kf= KFold(n_splits=5, shuffle=True, random_state=42)

In [0]:
X_train_linear= linear_prep.fit_transform(X_train)
X_test_linear= linear_prep.transform(X_test)
X_train_tree= tree_prep.fit_transform(X_train)
X_test_tree= tree_prep.transform(X_test)

1. RidgeClassifier
2. LogisticRegression 
3. ExtraTreesClassifier
4. RandomForestClassifier 
5. GradientBoostingClassifier
6. DecisionTreeClassifier

In [0]:
linear_pipe=Pipeline([
    ('model',RidgeClassifier())
])

tree_pipe=Pipeline([
    ('model', ExtraTreesClassifier())
])
linear_grid= [

    {
        'model':[RidgeClassifier()],
        'model__alpha':np.linspace(0.01,10,10),
        'model__max_iter':[100,200,300],
        'model__random_state':[42]
    },
    {
        'model':[LogisticRegression()],
        'model__penalty':['l2'],
        'model__max_iter':[5000],
        'model__random_state':[42]
    }
]

tree_grid= [
        {
        'model':[ExtraTreesClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    },
    {
        'model':[RandomForestClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    },
    {
        'model':[GradientBoostingClassifier()],
        'model__loss':['log_loss','exponential'],
        'model__learning_rate':np.linspace(0.01,1,5),
        'model__n_estimators':[100,200,300],
        'model__random_state':[42]
    },
    {
        'model':[DecisionTreeClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__random_state':[42],
        'model__ccp_alpha':[1e-4,1e-2,0.2]
    }
]

linear_search= GridSearchCV(linear_pipe, param_grid=linear_grid, cv=kf, scoring='f1', n_jobs=-1)
linear_search.fit(X_train_linear,y_train)
linear_estimator= linear_search.best_estimator_
print(linear_search.best_params_)

tree_search= GridSearchCV(tree_pipe, param_grid=tree_grid, cv=kf, scoring='f1', n_jobs=-1)
tree_search.fit(X_train_tree,y_train)
tree_estimator= tree_search.best_estimator_
print(tree_search.best_params_)

In [0]:
# linear_search = joblib.load('linear_estimator')
# tree_search = joblib.load('tree_estimator')

In [0]:
linear_features= linear_prep.get_feature_names_out()
linear_model= linear_estimator.named_steps['model']
linear_coef= linear_model.coef_[0]
linear_df= pd.DataFrame({
    'features':linear_features,
    'coef':linear_coef,
    'abs_coef':np.abs(linear_coef)
})

def get_group(feature_names):
    cleaned= feature_names.split("__")[-1]

    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

linear_df['group']= linear_df['features'].apply(get_group)
linear_df['importance_norm']= np.divide(linear_df['abs_coef'],linear_df['abs_coef'].sum())
linear_df= linear_df.sort_values(by='importance_norm', ascending=False)

In [0]:
linear_df.groupby('group')['importance_norm'].sum().plot(kind='bar', x='group', y='importance_norm')

In [0]:
tree_features= tree_prep.get_feature_names_out() 
tree_importances= tree_estimator.named_steps['model'].feature_importances_

In [0]:
tree_features

In [0]:
# Get tree feature importances
tree_features = tree_prep.get_feature_names_out()
tree_importances = tree_estimator.named_steps['model'].feature_importances_

# Create DataFrame
tree_df = pd.DataFrame({
    'features': tree_features,
    'importance': tree_importances,
})

# Extract feature groups
def get_group(feature_names):
    cleaned = feature_names.split("__")[-1]
    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

tree_df['group'] = tree_df['features'].apply(get_group)
tree_df['importance_norm'] = np.divide(tree_df['importance'], tree_df['importance'].sum())
tree_df = tree_df.sort_values(by='importance_norm', ascending=False)

display(tree_df.head(10))

In [0]:
# Visualize grouped feature importances for tree model
tree_df.groupby('group')['importance'].sum().sort_values(ascending=False).plot(
    kind='bar', 
    title='Feature Importance by Group (Tree Model)',
    xlabel='Feature Group',
    ylabel='Normalized Importance'
)

In [0]:
# joblib.dump(linear_estimator, 'linear_estimator')
# joblib.dump(tree_estimator, 'tree_estimator')

In [0]:
cm= metrics.confusion_matrix(y_test, tree_estimator.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(y_test, linear_estimator.predict(X_test_linear))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
print(linear_search.best_score_)
print(tree_search.best_score_)

In [0]:
tree_search.best_score_

In [0]:
linear_tree= tree_search.best_estimator_

In [0]:
linear_tree.fit(X_train_linear,y_train)

In [0]:
cm= metrics.confusion_matrix(y_test, linear_tree.predict(X_test_linear))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
lt_feature= linear_prep.get_feature_names_out()
lt_importance= linear_tree.named_steps['model'].feature_importances_

In [0]:
lt= pd.DataFrame({
    'features':lt_feature,
    'importance':lt_importance
})

lt['group']= lt['features'].apply(get_group)
lt['abs_importance']= np.divide(lt['importance'],lt['importance'].sum())

In [0]:
lt.groupby('group')['abs_importance'].sum().sort_values(ascending=False).plot(kind='bar', title='Feature Importances', figsize=(12,6))

In [0]:
from sklearn.ensemble import VotingClassifier

In [0]:
## Same Accuracy are linear_tree model

estimators= [
    ('le', linear_estimator),
    ('lt', linear_tree)
]
voting_model= VotingClassifier(estimators=estimators)
vote_pipe= Pipeline([
    ('model', voting_model)
])
voting_grid= [
    {
        'model':[voting_model],
        'model__voting':['hard', 'soft'],
        'model__weights':[(1,1)],
        'model__n_jobs':[-1]
    }
]
voting_search= GridSearchCV(vote_pipe, param_grid=voting_grid, cv=kf, scoring='f1')
voting_search.fit(X_train_linear, y_train)
print(voting_search.best_params_)
print(voting_search.best_score_)

In [0]:
cm= metrics.confusion_matrix(y_test, voting_search.best_estimator_.predict(X_test_linear))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
tree_search.score(X_test_linear, y_test)

In [0]:
# joblib.dump(linear_tree, 'linear_tree_model')

In [0]:
# linear_estimator= joblib.load('linear_estimator')
# linear_tree= joblib.load('linear_tree_model')
tree_estimator= joblib.load('tree_estimator')

In [0]:
X_test_data= linear_prep.transform(test_df.drop('is_fraud','trans_date_trans_time').toPandas())
y_test_data= test_df.toPandas()['is_fraud']

In [0]:
y_pred_data= linear_tree.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Linear Tree Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')


In [0]:
y_pred_data= voting_search.best_estimator_.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Voting Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
y_pred_data= linear_estimator.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Linear Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
X_test_data= tree_prep.transform(test_df.drop('is_fraud','trans_date_trans_time').toPandas())
y_test_data= test_df.toPandas()['is_fraud']

In [0]:
y_pred_data= tree_estimator.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Tree Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
importance= tree_estimator.named_steps['model'].feature_importances_
feature= tree_prep.get_feature_names_out()

In [0]:
tree_imp= pd.DataFrame({
    'feature':feature,
    'importance':importance
    })

def get_group(feature_names):
    cleaned= feature_names.split("__")[-1]

    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

tree_imp['feature']= tree_imp['feature'].apply(get_group)
tree_imp['abs_importance']= np.divide(tree_imp['importance'], tree_imp['importance'].sum())

In [0]:
tree_imp.groupby('feature')['abs_importance'].sum().plot(kind='bar')

In [0]:
boosting= xgb.XGBClassifier(random_state=42)

boost_pipe= Pipeline([
    ('model',boosting)
])

gbt_grid= {
        'model__n_estimators':[100,200,300,400],
        'model__max_depth':[3,5,10,15],
        'model__reg_lambda':[5],
        'model__objective':['binary:logistic'],
        'model__eval_metric':['logloss']
        # 'model__random_state':42,
    }


clf_gbt= GridSearchCV(boost_pipe, param_grid=gbt_grid, cv=kf, scoring='f1', n_jobs=-1)
# clf_gbt.fit(X_train_linear, y_train)
clf_gbt.fit(X_train_tree, y_train)
print(clf_gbt.best_params_)
print(clf_gbt.best_score_)

In [0]:
# clf_gbt= joblib.load('xgb_estimator')

In [0]:
y_pred= clf_gbt.predict(X_test_linear)
cm= metrics.confusion_matrix(y_test, y_pred)
# disp= metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
y_pred= clf_gbt.predict(X_test_tree)
cm= metrics.confusion_matrix(y_test, y_pred)
# disp= metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost Model (Tree Prep)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
y_pred_data= clf_gbt.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
# disp= metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost Model (linear Prep)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
y_pred_data= clf_gbt.predict(X_test_data)
cm= metrics.confusion_matrix(y_test_data, y_pred_data)
# disp= metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost Model (tree Prep)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
linear_importance= clf_gbt.named_steps['model'].feature_importances_
linear_feature= linear_prep.get_feature_names_out()

In [0]:
gbt_linear= pd.DataFrame({
    'feature':linear_feature,
    'importance':linear_importance
    })

def get_group(feature_names):
    cleaned= feature_names.split("__")[-1]

    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

gbt_imp['feature']= gbt_linear['feature'].apply(get_group)
gbt_linear['abs_importance']= np.divide(gbt_linear['importance'], gbt_linear['importance'].sum())

In [0]:
gbt_linear.groupby('feature')['abs_importance'].sum().plot(kind='bar')

In [0]:
tree_importance= clf_gbt.best_estimator_.named_steps['model'].feature_importances_
tree_feature= tree_prep.get_feature_names_out()

In [0]:
gbt_tree= pd.DataFrame({
    'feature':tree_feature,
    'importance':tree_importance
    })

def get_group(feature_names):
    cleaned= feature_names.split("__")[-1]

    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

gbt_tree['feature']= gbt_tree['feature'].apply(get_group)
gbt_tree['abs_importance']= np.divide(gbt_tree['importance'], gbt_tree['importance'].sum())

In [0]:
gbt_tree.groupby('feature')['abs_importance'].sum().plot(kind='bar')

In [0]:
# joblib.dump(clf_gbt.best_estimator_,'xgb_estimator')

In [0]:
## Required features
[
    'amt',
    'category',
    'hour',
    'merchant',
    'city',
    'merch',
    'current_age',
    'job',
    'state'
]

In [0]:
joblib.dump(clf_gbt.best_estimator_,'xgb_tree_prep_estimator')